In [2]:
from orderbike.plot import plot_graph
import geopandas as gpd
from matplotlib import pyplot as plt
import networkx as nx
import shapely
import momepy

/Users/clse/miniforge3/envs/orderbike/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [14]:
folderoots = "../data/processed/ignored_files/paper/smrainet/"
streets = gpd.read_file(folderoots + "idf_metro.gpkg")
streets = streets.loc[streets.geometry.type == "LineString"]
streets = streets.to_crs(streets.estimate_utm_crs())
G = momepy.gdf_to_nx(streets, approach="primal", integer_labels=True)
for e in G.edges:
    G.edges[e]["length"] = G.edges[e]["geometry"].length
closeness = nx.closeness_centrality(G, distance="length")
edge_closeness = {}
for edge in G.edges:
    edge_closeness[edge] = (closeness[edge[0]] + closeness[edge[1]]) / 2
init_edges = [tuple(max(edge_closeness, key=edge_closeness.get))]
fig_ind, ax_ind = plot_graph(
    G,
    filepath=folderoots + "/plots",
    figsize=(12, 12),
    buffer=False,
    buff_size=152,
    buff_color="grey",
    edge_color="black",
    edge_linewidth=4,
    node_color="black",
    node_size=200,
    show=False,
    save=False,
    close=False,
)

gdf_edge_ind = gpd.GeoDataFrame(geometry=[G.edges[init_edges[0]]["geometry"]])
geom_node = [shapely.Point(G.nodes[n]["x"], G.nodes[n]["y"]) for n in init_edges[0][:2]]
gdf_edge = gpd.GeoDataFrame(
    geometry=list(nx.get_edge_attributes(G, "geometry").values())
)
gdf_node = gpd.GeoDataFrame(geometry=geom_node)
gdf_node.plot(ax=ax_ind, color="#FF0000", zorder=3, markersize=200)
gdf_edge_ind.plot(ax=ax_ind, zorder=3, color="#FF0000", linewidth=4)
savename = folderoots + "/plots/netviz.png"
ax_ind.axis("off")
fig_ind.savefig(savename)
plt.close()